# 03 — Evaluate: score runs, tune the abstain threshold, read errors, check the data layer

1. Score runs on validation (PA, per answer type, per shop) and compare versions on the same ids
2. τ sweep from logged samples (no LLM calls) and the errors worth reading
3. Rule-retrieval study: lexical vs bi-encoder vs hybrid on hand-labelled train questions
4. Normalization check: fixed-code monthly revenue over the normalized tables vs train answers

Validation means train questions (`splits/val300.txt`); tune τ there, never on the public test.
Train has none of the public test's phone / repair / mother-and-baby shops (21% of public questions).

In [ ]:
RUNS = ["v3_train"]                 # folders under runs/ (or absolute paths) produced by 02_agent
IDS = "../splits/val300.txt"        # validation ids; None scores every logged question
RETRIEVAL_STUDY = False             # §3 loads two embedders on CPU (a few minutes)
NORMALIZATION_CHECK = True          # §4 reads every train shop (cached after the first run)

In [ ]:
from datamate.nblib import run_lib_cells
from datamate.scoring import penalized_accuracy, score_one

run_lib_cells("02_agent.ipynb", globals())
pd.set_option("display.max_colwidth", 120, "display.width", 200)

REFS = {r["id"]: r for r in load_answers("train").to_dict("records")}
QS = load_questions("train").set_index("id")
VAL = set(Path(IDS).read_text().split()) if IDS else None


def run_dir(name) -> Path:
    p = Path(name)
    return p if p.is_absolute() else REPO / "runs" / name


def load_logs(run: Path) -> dict[str, dict]:
    return {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in (run / "logs").glob("*.json")}

## 1. Scores

In [ ]:
def score_run(run: Path, ids: set[str] | None) -> tuple[dict, pd.DataFrame]:
    preds = {r["id"]: r for r in read_jsonl(run / "submission.jsonl")}
    ids = (ids & set(preds)) if ids else set(preds)
    refs = {i: REFS[i] for i in ids if i in REFS}
    res = penalized_accuracy(preds, refs)
    df = pd.DataFrame([{"id": k, "score": v, "ref_type": refs[k]["type"], "pred_type": preds[k]["type"],
                        "business_id": QS.loc[k, "business_id"], "question": QS.loc[k, "question"]}
                       for k, v in res.pop("per_question").items()])
    return res, df


def summarize(df: pd.DataFrame, by: str) -> pd.DataFrame:
    g = df.groupby(by).score
    return pd.DataFrame({"n": g.size(), "PA": g.mean().round(3), "correct": g.apply(lambda s: (s == 1).sum()),
                         "wrong": g.apply(lambda s: (s == -1).sum()), "abstain": g.apply(lambda s: (s == 0).sum())})


SCORES = {name: score_run(run_dir(name), VAL) for name in RUNS}
pd.DataFrame({name: {k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()}
              for name, (res, _) in SCORES.items()}).T

In [ ]:
res, df = SCORES[RUNS[-1]]
summarize(df, "ref_type")

In [ ]:
pd.crosstab(df.ref_type, df.pred_type, margins=True)

In [ ]:
summarize(df, "business_id").sort_values("PA").head(15)

### Versions on the same questions
Runs may cover different subsets (a run stopped early); compare only on their common ids.

In [ ]:
common = set.intersection(*(set(SCORES[n][1].id) for n in RUNS)) if RUNS else set()
pd.DataFrame({n: SCORES[n][1].set_index("id").loc[sorted(common)].score.agg(
    PA="mean", correct=lambda s: (s == 1).sum(), wrong=lambda s: (s == -1).sum()) for n in RUNS}).T.assign(n=len(common))

## 2. Abstain threshold and errors
Each log keeps every sample's answer, so the vote is recomputed at any τ without the LLM.
τ = 0 shows raw precision (always answer the top cluster): report it next to the tuned PA, since an
abstain gate can hide a prompt that made answers worse (v0 → v1).

In [ ]:
def sweep_tau(run: Path, ids, taus=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0)) -> pd.DataFrame:
    logs = load_logs(run)
    k = json.loads((run / "config.json").read_text())["k"]
    ids = [i for i in (ids or logs) if i in logs and i in REFS]
    votes = {i: vote([postprocess(logs[i]["question"], s.get("answer")) for s in logs[i]["samples"]
                      if not s.get("repair")], k) for i in ids}
    rows = []
    for tau in taus:
        s = pd.Series([score_one(v["answer"] if v["answer"] is not None and v["support"] >= tau else None, REFS[i])
                       for i, v in votes.items()])
        answered = int((s != 0).sum())
        rows.append({"tau": tau, "PA": round(s.mean(), 4), "answered": answered,
                     "precision": round((s == 1).sum() / answered, 3) if answered else float("nan")})
    return pd.DataFrame(rows)


sweep_tau(run_dir(RUNS[-1]), VAL)

In [ ]:
LOGS = load_logs(run_dir(RUNS[-1]))


def show(qid, max_code=2500):
    lg, ref = LOGS[qid], REFS[qid]
    print(lg["business_id"], lg["as_of_date"], "|", lg["question"])
    print("REF :", ref["type"], json.dumps(ref.get("answer"), ensure_ascii=False)[:200])
    print("PRED:", json.dumps(lg["final"], ensure_ascii=False)[:200], "| support", lg.get("support"))
    for i, s in enumerate(lg["samples"]):
        print(f"--- sample {i}: answer={json.dumps(s.get('answer'), ensure_ascii=False)[:120]} "
              f"error={str(s.get('error'))[-200:]}")
        print((s.get("code") or "")[:max_code])
        print(">>> stdout:", (s.get("stdout") or "")[-600:])


wrong = df[df.score == -1]
print(len(wrong), "wrong")
wrong[["id", "business_id", "ref_type", "pred_type", "question"]].head(30)

In [ ]:
for qid in wrong.id.head(3):
    show(qid)
    print("=" * 100)

## 3. Rule-retrieval study
Labels: `splits/rule_labels_train.json`, 48 val300 questions hand-labelled (by Claude) with the train
rulebook rows that change how each is computed (§9 format rows excluded). Small, single annotator:
differences under ~5 points are noise. Recall@k: share of a question's rows in the top k;
Precision@k: share of the top k that are labelled (low precision = distracting context).
Last measured: lexical R@3 0.41 P@3 0.31 R@5 0.47; Qwen3-Embedding-0.6B 0.27 / 0.19 / 0.31;
bge-m3 0.29 / 0.21 / 0.36; hybrid lexical + bge-m3 0.36 / 0.26 / 0.41.

In [ ]:
if RETRIEVAL_STUDY:
    rulebook = (split_root("train") / "rulebook.md").read_text(encoding="utf-8")
    labels = json.loads((REPO / "splits" / "rule_labels_train.json").read_text())["labels"]
    lex = LexicalRuleRetriever(rulebook)
    names = [r.name for r in lex.rules]
    gold = {qid: {names.index(n) for n in ns} for qid, ns in labels.items()}
    questions = QS.loc[list(labels)].question.tolist()
    rankings = {"lexical": [lex.rank(q) for q in questions]}
    for model in ["Qwen/Qwen3-Embedding-0.6B", "BAAI/bge-m3"]:
        short = model.split("/")[1]
        dense = DenseRuleRetriever(rulebook, model)
        rankings[short] = [dense.rank(q) for q in questions]
        rankings[f"hybrid lexical+{short}"] = [rrf([a, b]) for a, b in zip(rankings["lexical"], rankings[short])]

    def metrics(ranks, ks=(1, 3, 5, 8)):
        out = {}
        for k in ks:
            out[f"R@{k}"] = round(sum(len(set(r[:k]) & gold[q]) / len(gold[q]) for q, r in zip(gold, ranks)) / len(gold), 3)
            out[f"P@{k}"] = round(sum(len(set(r[:k]) & gold[q]) / k for q, r in zip(gold, ranks)) / len(gold), 3)
        return out

    display(pd.DataFrame({name: metrics(r) for name, r in rankings.items()}).T)

## 4. Normalization check
For plain "doanh thu tháng M/Y" questions, revenue computed by fixed code over the normalized
tables (rulebook §1: drop the whole order if cancelled or with any returned line; §8: identical lines
of one order count once) must equal the reference. A mismatch is a normalization bug or a rule this
oracle does not model. Last run: 71 / 71 on questions whose answer is total-revenue sized; the
other "mismatches" are per-product revenue questions the keyword filter lets through.

In [ ]:
REVENUE = re.compile(r"(doanh thu|dthu)")
MONTH = re.compile(r"thang (\d{1,2})(?: nam |/)(\d{4})")
NOT_PLAIN = re.compile(r"nhom|khach|mat hang|\bsp\b|%|phan tram|trung binh|binh quan|so voi|tang|giam|chenh|ngay nao"
                       r"|\bdon\b|\bca\b|tuan|tien mat|chuyen khoan|cod|kenh|hoat chat|size|topping|\bly\b|top|nhat"
                       r"|quy|lien|moi thang|tu thang|den thang|\bva\b|hay")
CANCEL = re.compile(r"huy|cancel|khong giao duoc")
RETURN = re.compile(r"tra hang|tra lai|hoan tra|hang tra ve|doi tra|hoan hang|return")


def line_table(biz):
    """The order-line view: README meanings for order id, sale date and line amount (status, qty if any)."""
    def find(roles, prefix):
        return next((c for c, r in roles.items() if fold(r).startswith(prefix)), None)
    best = None
    for name in main_keys(biz):
        roles = biz.roles.get(name, {})
        cols = {k: find(roles, p) for k, p in [("id", "ma don"), ("date", "ngay ban"), ("status", "trang thai don"),
                                               ("amount", "thanh tien sau giam"), ("qty", "so luong")]}
        if all(cols[k] for k in ("id", "date", "amount")):
            best = best or (name, cols)
    return best


def monthly_revenue(biz, month, year):
    found = line_table(biz)
    if found is None:
        return None, "no line table with README meanings"
    name, c = found
    df = biz.frames[name]
    raw = [col for col in df.columns if "__" not in str(col)] + ([SOURCE_COL] if SOURCE_COL in df else [])
    df = df.drop_duplicates(subset=raw)
    status = df[c["status"]].map(fold) if c["status"] else pd.Series("", index=df.index)
    bad = status.str.contains(CANCEL) | status.str.contains(RETURN)
    if c["qty"]:
        qty = df[c["qty"] + NUM_SUFFIX] if c["qty"] + NUM_SUFFIX in df else pd.to_numeric(df[c["qty"]], errors="coerce")
        bad |= qty < 0
    d = df[c["date"] + DATE_SUFFIX]
    amt = df[c["amount"] + NUM_SUFFIX] if c["amount"] + NUM_SUFFIX in df else pd.to_numeric(df[c["amount"]], errors="coerce")
    keep = (d.dt.month == month) & (d.dt.year == year) & ~df[c["id"]].isin(set(df.loc[bad, c["id"]]))
    return int(round(amt[keep].sum())), name


if NORMALIZATION_CHECK:
    Q = load_questions("train").merge(load_answers("train"), on="id")

    def plain_month(q):
        f = fold(q)
        m = MONTH.search(f)
        return (int(m[1]), int(m[2])) if REVENUE.search(f) and m and not NOT_PLAIN.search(f) else None

    Q["month"] = Q.question.map(plain_month)
    rows, cache = [], {}
    for _, r in Q[Q.month.notna() & (Q.type == "number")].iterrows():
        cache.setdefault(r.business_id, load_business_cached("train", r.business_id))
        got, how = monthly_revenue(cache[r.business_id], *r.month)
        rows.append(dict(biz=r.business_id, month=r.month, ref=r.answer, got=got, ok=got == r.answer, table=how))
    CHK = pd.DataFrame(rows)
    sized = CHK[CHK.got.notna() & (CHK.ref.astype(float) > 1e6)]
    print(f"total-revenue sized answers matched: {int(sized.ok.sum())} / {len(sized)}"
          f" | shops without a README-described line table: {CHK[CHK.got.isna()].biz.nunique()}")
    display(CHK.groupby("biz").agg(n=("ok", "size"), match=("ok", "sum"), table=("table", "first")))